<a href="https://www.kaggle.com/code/likhitakoppuravuri/prophetmodelver?scriptVersionId=282638330" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/crp-yield/scikitlearn/default/1/crpDataSet_avg0.csv


In [2]:
!pip install prophet
from prophet import Prophet

In [3]:
import pandas as pd
from prophet import Prophet

# Load data with date parsing
data = pd.read_csv('/kaggle/input/crp-yield/scikitlearn/default/1/crpDataSet_avg0.csv', dayfirst=True, parse_dates=['Date'])
#data = pd.read_csv('crpDataSet_avg0.csv', dayfirst=True, parse_dates=['Date'])

# Rename columns for Prophet
df = data.rename(columns={'Date': 'ds', 'Crop_Yield': 'y'})

# Define the split date for train/test
split_date = pd.to_datetime('2018-04-01')  # example split date

# Split data
train = df[df['ds'] < split_date]
test = df[df['ds'] >= split_date]

# Fit Prophet Model
model = Prophet()
model.fit(train)

# Create future dates dataframe for prediction
future = model.make_future_dataframe(periods=len(test), freq='D')
future = future[future['ds'] >= split_date]  # only future = test period

# Predict crop yield
forecast = model.predict(future)
# Extract predictions
prophet_forecast = forecast['yhat'].values

# Now you can assemble results DataFrame with actual and forecasted values for error metric computations
results = pd.DataFrame({
    'Actual': test['y'].values,
    'Prophet': prophet_forecast
}, index=test['ds'])

# Drop NaNs and compute errors as before
results = results.dropna()
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
mae = mean_absolute_error(results['Actual'], results['Prophet'])
rmse = np.sqrt(mean_squared_error(results['Actual'], results['Prophet']))

print(f"Prophet: MAE={mae:.4f}, RMSE={rmse:.4f}")

13:08:28 - cmdstanpy - INFO - Chain [1] start processing
13:08:29 - cmdstanpy - INFO - Chain [1] done processing


Prophet: MAE=7.5009, RMSE=9.9227
